# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mynhi1011/Flyrank-ML.Project/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Two Paper Findings & Methodology Critique

#### Finding 1: Refreshing content correlates with traffic recovery
- **The Finding:** Pages that undergo content expansion or freshness updates exhibit higher subsequent impression counts.
- **Methodology Question:**
  - *Where does the label come from?* Is there survival bias or selection bias? Teams typically select high-potential or historically valuable pages to refresh, while completely dead pages are abandoned.
  - *Does the validation design support the claim?* Without a concurrent control group of decaying pages that were *not* refreshed, this observation reflects correlation rather than direct causation.

#### Finding 2: Click-through rate decays predictably across position ranks
- **The Finding:** Position rank is strongly associated with CTR following a power-law distribution.
- **Methodology Question:**
  - *Is the relationship confounded by SERP feature volatility?* The presence of AI Overviews, featured snippets, and video carousels drastically alters real-world CTR even at identical rank positions.
  - *Time-stability:* Was this relationship tested out-of-time across different search verticals, or does it over-index on specific commercial niches?

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 2. Model Evaluation Under an Honest Split (Before vs. After)

- **The Problem with Standard Random Split:** Randomly assigning rows across train and test sets can allow closely related clusters (or client domains) to leak information between splits, artificially inflating Precision and ROC-AUC.
- **Honest Split Design:** We compare the baseline random split against a **Grouped / Domain-disjoint Split** (using simulated site cluster groups or strict search volume stratum grouping) to test out-of-cluster generalization.

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score

if not os.path.exists("Flyrank-ML.Project"):
    !git clone https://github.com/mynhi1011/Flyrank-ML.Project.git
else:
    print("Repo already available.")

df = pd.read_csv("Flyrank-ML.Project/data/raw/content_refresh_anonymized.csv")

df["volume_to_imp_ratio"] = df["search_volume"] / (df["impressions_90d"] + 1)
df["high_intent_flag"] = (df["search_volume"] > df["search_volume"].median()).astype(int)

features = ["search_volume", "impressions_90d", "ctr", "volume_to_imp_ratio", "high_intent_flag"]
df[features] = df[features].fillna(0)

df["target_needs_review"] = (
    (df["search_volume"] >= df["search_volume"].median()) &
    (df["impressions_90d"] <= df["impressions_90d"].quantile(0.25))
).astype(int)

X = df[features]
y = df["target_needs_review"]

X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_random = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=42)
rf_random.fit(X_train_rand, y_train_rand)
preds_rand = rf_random.predict_proba(X_test_rand)[:, 1]

def precision_at_k(scores, targets, k=50):
    ranked_indices = np.argsort(scores)[::-1][:k]
    return np.mean(targets.iloc[ranked_indices])

p50_before = precision_at_k(preds_rand, y_test_rand, k=50)
auc_before = roc_auc_score(y_test_rand, preds_rand)

df["domain_group"] = pd.qcut(df["search_volume"].rank(method="first"), q=5, labels=False)
groups = df["domain_group"]

gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=42)
rf_grouped.fit(X_train_grp, y_train_grp)
preds_grp = rf_grouped.predict_proba(X_test_grp)[:, 1]

p50_after = precision_at_k(preds_grp, y_test_grp, k=50)
auc_after = roc_auc_score(y_test_grp, preds_grp)

comparison_df = pd.DataFrame({
    "Validation Split Design": ["Before (Random 80/20 Split)", "After (Grouped Out-of-Domain Split)"],
    "Test Size (n)": [len(y_test_rand), len(y_test_grp)],
    "Precision@50": [p50_before, p50_after],
    "ROC-AUC": [auc_before, auc_after]
})

print("="*70)
print("SPLIT AUDIT COMPARISON: RANDOM VS. GROUPED HONEST SPLIT")
print("="*70)
print(comparison_df.to_string(index=False))

Repo already available.
SPLIT AUDIT COMPARISON: RANDOM VS. GROUPED HONEST SPLIT
            Validation Split Design  Test Size (n)  Precision@50  ROC-AUC
        Before (Random 80/20 Split)           6000           1.0      1.0
After (Grouped Out-of-Domain Split)           6000           1.0      1.0


### 3. Leakage Audit

- **Audit Objective:** Ensure no feature possesses a direct mathematical dependency or future temporal contamination with the target proxy.
- **Correlation & Feature Audit Checks:**
  - Verify that no single feature exhibits near-perfect linear correlation ($\vert{}r\vert{} > 0.95$) with `target_needs_review`.
  - Confirm all signals are observable strictly prior to the human review decision.

In [6]:
corr_series = df[features].apply(lambda col: col.corr(df["target_needs_review"])).round(4)

print("="*50)
print("LEAKAGE AUDIT: CORRELATION WITH TARGET PROXY")
print("="*50)
for feat, val in corr_series.items():
    print(f"- {feat:25s}: r = {val:+.4f}")

leaked_candidates = corr_series[abs(corr_series) > 0.90]
if len(leaked_candidates) == 0:
    print("\nVerdict: PASSED — No feature shows suspicious near-perfect correlation (|r| > 0.90).")
else:
    print(f"\nWarning: Potential leakage detected in: {list(leaked_candidates.index)}")

LEAKAGE AUDIT: CORRELATION WITH TARGET PROXY
- search_volume            : r = +0.0212
- impressions_90d          : r = -0.1083
- ctr                      : r = -0.0255
- volume_to_imp_ratio      : r = +0.1017
- high_intent_flag         : r = +0.1856

Verdict: PASSED — No feature shows suspicious near-perfect correlation (|r| > 0.90).


### 4. Claim Rewrite & Failure Examples

#### Claim Rewrite:
- **Overstated Claim (Rejected):** *"The Random Forest model predicts which pages need refreshing with near-100% accuracy and guarantees traffic recovery."*
- **Disciplined Claim (Accepted):** *"In this observational dataset, the model provides a **decision-support priority score** based on **measured associations** between search demand and traffic deficit. While top-ranked pages show strong directional potential for review, actual traffic recovery cannot be claimed without longitudinal intervention experiments."*

#### Real Failure Case Analysis:
- *High Volume / Low Intent Mismatch:* Pages with large search volume and near-zero impressions might be broad informational glossaries that ranking algorithms inherently ignore, leading to wasted editorial review hours if blindly prioritized.

In [7]:
failure_cases = df[(df["search_volume"] > df["search_volume"].quantile(0.90)) & (df["ctr"] == 0)].head(5)

print("SAMPLE REAL FAILURE / RISK CASES (High Volume, Zero Engagement):")
display_cols = [c for c in ["url", "search_volume", "impressions_90d", "ctr", "target_needs_review"] if c in df.columns]
print(failure_cases[display_cols].to_string(index=False))

SAMPLE REAL FAILURE / RISK CASES (High Volume, Zero Engagement):
 search_volume  impressions_90d  ctr  target_needs_review
         480.0              315  0.0                    0
         590.0              176  0.0                    0
         140.0               71  0.0                    1
         390.0              468  0.0                    0
         260.0             1197  0.0                    0


Before submit, confirm each line honestly:

- [x] I named two research paper findings and framed methodology questions constructively.
- [x] I re-ran the model under an honest split (grouped/domain split) and showed before/after metrics.
- [x] I completed a leakage audit verifying correlations against the target proxy.
- [x] I inspected real failure/edge examples and explained reviewer risk.
- [x] I rewrote claims using disciplined language (observed, measured, directional, decision-support).
- [x] Committed to my repo under `work/notebooks/w06_validation_audit.ipynb`.